# 📖 Notebook 5: Queries, Heartbeats, Continue-As-New, and Testing

This notebook closes the loop on four concepts that every production Temporal user needs:

| Concept | In plain English | Real-world example |
|---|---|---|
| **Queries** | Read the current state of a running workflow | "What step is this order on right now?" |
| **Heartbeats** | Long activities prove they're still alive | A 30-min video encode that tells Temporal "still working" every 10s |
| **Continue-As-New** | Reset a long-running workflow's history to keep it fast | A daily report workflow that runs forever |
| **Testing** | Run workflows in a fake Temporal server — no Docker needed | Unit tests in CI |

## Why these matter

- **Signals** (notebook 4) push data *into* a workflow. **Queries** pull data *out*.
- Without **heartbeats**, a frozen long-running activity looks identical to a crashed one.
- Without **Continue-As-New**, workflows that loop forever eventually hit history-size limits and crash.
- Without **Testing**, you can't safely refactor workflows — you're flying blind.

## Learning Objectives

- Add a `@workflow.query` method to expose live state
- Use `activity.heartbeat()` to report progress and recover from activity crashes
- Use `workflow.continue_as_new()` for eternal workflows
- Write a unit test using `WorkflowEnvironment` — no Docker required

## 🛠️ Setup

Make sure Temporal is running:

```bash
cd 03-technologies/workflow-engines/temporal
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import asyncio
import uuid
from datetime import timedelta
from temporalio import activity, workflow
from temporalio.client import Client
from temporalio.worker import Worker
from temporalio.common import RetryPolicy

client = await Client.connect("localhost:7233")
TASK_QUEUE = "pro-task-queue"
print("✅ Connected to Temporal")

---
## 🔎 Part 1: Queries — reading live workflow state

### The problem

Your order workflow is running. A customer calls support asking "Where is my order?" The support engineer needs to know **right now** which step the workflow is on.

Without queries, you'd have to dig through event history or update an external database after every step. Both are slow and error-prone.

### The solution: `@workflow.query`

A **query** is a read-only method on your workflow. You can call it from outside at any time — while the workflow is running, sleeping, or waiting for a signal. Queries must be **side-effect free** (they're like `SELECT`, not `UPDATE`).

```
Signals (write) ─▶ Workflow ◀─ Queries (read)
```

In [ ]:
# A workflow that processes an order in 3 steps and exposes its progress via a query.

@activity.defn
async def do_step(name: str) -> str:
    activity.logger.info(f"Doing step: {name}")
    await asyncio.sleep(1)  # simulate work
    return f"{name}-done"


@workflow.defn
class TrackableOrderWorkflow:
    """Runs 3 steps and lets callers ask 'what step are you on?' anytime."""

    def __init__(self):
        self.current_step = "not_started"
        self.completed = []

    # Queries: read-only views of state. No activities, no side effects.
    @workflow.query
    def get_status(self) -> dict:
        return {"current_step": self.current_step, "completed": list(self.completed)}

    @workflow.query
    def get_progress_percent(self) -> int:
        return int(len(self.completed) / 3 * 100)

    @workflow.run
    async def run(self, order_id: str) -> dict:
        for step in ["validate", "charge", "ship"]:
            self.current_step = step
            await workflow.execute_activity(
                do_step, step,
                start_to_close_timeout=timedelta(seconds=10),
            )
            self.completed.append(step)
        self.current_step = "done"
        return {"order_id": order_id, "completed": self.completed}


print("✅ Defined TrackableOrderWorkflow with 2 queries: get_status, get_progress_percent")

In [ ]:
# NOTE: Temporal normally runs workflow code in a sandbox that re-imports the
# defining module. In a notebook that module is `__main__`, so the re-import
# re-runs these cells and fails with "Failed validating workflow ..." (caused by
# "asyncio.run() cannot be called from a running event loop"). Notebooks must use
# UnsandboxedWorkflowRunner. In a real worker process keep the default sandbox --
# it is what protects you from non-deterministic imports.
from temporalio.worker import UnsandboxedWorkflowRunner
# Run the workflow and query it while it's still working.

async def demo_queries():
    async with Worker(
        client,
        task_queue=TASK_QUEUE,
        workflows=[TrackableOrderWorkflow],
        activities=[do_step],
        workflow_runner=UnsandboxedWorkflowRunner(),
    ):
        wf_id = f"trackable-{uuid.uuid4()}"
        handle = await client.start_workflow(
            TrackableOrderWorkflow.run,
            "ORD-999",
            id=wf_id,
            task_queue=TASK_QUEUE,
        )

        # Poll the workflow while it runs. Each query is instant — no polling of a DB!
        for _ in range(4):
            await asyncio.sleep(0.8)
            status = await handle.query(TrackableOrderWorkflow.get_status)
            pct = await handle.query(TrackableOrderWorkflow.get_progress_percent)
            print(f"   📊 {pct:>3}%  current={status['current_step']:<12}  done={status['completed']}")

        result = await handle.result()
        return result


result = await demo_queries()
print(f"\n📬 Final: {result}")
print("\n💡 The support dashboard can call get_status() to show live progress")
print("   — no extra database writes, no stale data.")

### Queries vs Signals — quick reference

| | Signal (`@workflow.signal`) | Query (`@workflow.query`) |
|---|---|---|
| Direction | Outside → Workflow | Workflow → Outside |
| Purpose | Change state / give input | Read state |
| Side effects | ✅ Allowed (sets fields) | ❌ Forbidden (read-only) |
| Can call activities | ❌ No | ❌ No |
| Recorded in history | ✅ Yes | ❌ No (query runs on-demand) |
| API | `handle.signal(...)` | `handle.query(...)` |

---
## 💓 Part 2: Heartbeats — keeping long activities alive

### The problem

An activity encodes a video. It takes 30 minutes. Halfway through, the worker machine dies.

- **Without heartbeats**: Temporal can't tell the difference between "taking a long time" and "crashed". It must wait for the full `start_to_close_timeout` (say 1 hour) before retrying. Meanwhile, other workers sit idle.
- **With heartbeats**: The activity calls `activity.heartbeat(progress)` every few seconds. If the heartbeat stops, Temporal knows the activity died and retries it on another worker almost immediately.

Heartbeats also let an activity **resume from where it left off** after a retry — you pass a progress token in the heartbeat, and the retried activity reads it via `activity.info().heartbeat_details`.

### The two timeouts

| Timeout | What it means |
|---|---|
| `start_to_close_timeout` | Total wall-clock budget for one activity attempt |
| `heartbeat_timeout` | Max time allowed between heartbeats — if exceeded, activity is considered crashed |

In [ ]:
# A long-running activity that heartbeats its progress.

@activity.defn
async def encode_video(video_id: str) -> dict:
    total_chunks = 10

    # If this is a retry, pick up from the last reported heartbeat.
    start_chunk = 0
    if activity.info().heartbeat_details:
        start_chunk = activity.info().heartbeat_details[0]
        activity.logger.info(f"Resuming from chunk {start_chunk}")

    for i in range(start_chunk, total_chunks):
        await asyncio.sleep(0.2)  # simulate encoding one chunk
        activity.heartbeat(i + 1)  # tell Temporal 'I've done i+1 chunks'
        activity.logger.info(f"Encoded chunk {i + 1}/{total_chunks}")

    return {"video_id": video_id, "chunks": total_chunks}


@workflow.defn
class VideoEncodingWorkflow:
    @workflow.run
    async def run(self, video_id: str) -> dict:
        return await workflow.execute_activity(
            encode_video,
            video_id,
            start_to_close_timeout=timedelta(minutes=30),
            # If no heartbeat for 5s, assume the worker crashed and retry.
            heartbeat_timeout=timedelta(seconds=5),
            retry_policy=RetryPolicy(maximum_attempts=3),
        )


print("✅ Defined heartbeating encode_video activity and VideoEncodingWorkflow")

In [ ]:
# NOTE: Temporal normally runs workflow code in a sandbox that re-imports the
# defining module. In a notebook that module is `__main__`, so the re-import
# re-runs these cells and fails with "Failed validating workflow ..." (caused by
# "asyncio.run() cannot be called from a running event loop"). Notebooks must use
# UnsandboxedWorkflowRunner. In a real worker process keep the default sandbox --
# it is what protects you from non-deterministic imports.
from temporalio.worker import UnsandboxedWorkflowRunner
async def demo_heartbeat():
    async with Worker(
        client,
        task_queue=TASK_QUEUE,
        workflows=[VideoEncodingWorkflow],
        activities=[encode_video],
        workflow_runner=UnsandboxedWorkflowRunner(),
    ):
        return await client.execute_workflow(
            VideoEncodingWorkflow.run,
            "video-42",
            id=f"encode-{uuid.uuid4()}",
            task_queue=TASK_QUEUE,
        )


result = await demo_heartbeat()
print(f"📬 Result: {result}")
print("\n💡 In the Temporal UI, open this workflow and look at the activity's")
print("   'Pending Activity' panel — you can see live heartbeat progress.")
print("   If the worker had crashed, Temporal would have restarted encoding")
print("   from the last reported chunk instead of from chunk 0.")

### When to heartbeat

| Activity length | Heartbeat? |
|---|---|
| < 5 seconds | No — not worth it |
| 5 s – 1 min | Optional, but helps detect hangs |
| > 1 min | **Always** — otherwise crashes waste a lot of time |
| Has progress you can resume from | **Yes** — pass the progress value to `heartbeat()` |

---
## 🔁 Part 3: Continue-As-New — workflows that live forever

### The problem

Some workflows run forever:
- A **daily report** workflow that runs once a day, every day
- A **subscription** workflow that charges monthly for years
- A **device agent** workflow that stays alive for the life of the device

Every step in a workflow appends to its event history. After ~50,000 events or ~50 MB, Temporal rejects new events — it's **too big to replay efficiently**.

### The solution: `workflow.continue_as_new()`

Think of it like a fresh restart for the same logical workflow. The **workflow ID stays the same** from the outside, but internally Temporal closes the old execution and starts a new one with a clean history. State you want to keep is passed as an argument.

```
Original execution (events 1-5000)
         │
         └─▶ continue_as_new(new_state)
                       │
                       └─▶ New execution (events 1-5000)
                                    │
                                    └─▶ continue_as_new(...)   ...and so on, forever.
```

In [ ]:
# A periodic 'cron-like' workflow that runs forever using continue_as_new.
# Every iteration: do some work, sleep, then continue-as-new with updated state.

@activity.defn
async def run_daily_report(day: int) -> dict:
    activity.logger.info(f"Generating report for day {day}")
    await asyncio.sleep(0.2)
    return {"day": day, "rows": 1000 + day}


@workflow.defn
class DailyReportWorkflow:
    """Runs a daily report, then continues-as-new for the next day.

    For the demo we stop after 3 iterations and use a 0.5s 'day'.
    In production: sleep for 24 hours and loop forever.
    """

    @workflow.run
    async def run(self, day: int = 1, max_days: int = 3) -> dict:
        # Do one day's work
        result = await workflow.execute_activity(
            run_daily_report, day,
            start_to_close_timeout=timedelta(seconds=10),
        )
        workflow.logger.info(f"Report day {day}: {result}")

        if day >= max_days:
            # Demo only — normally we'd never stop
            return {"finished_at_day": day, "last_result": result}

        # Sleep until 'tomorrow'. In prod: timedelta(days=1).
        await workflow.sleep(timedelta(seconds=0.5))

        # Restart with a clean history, carrying the day counter forward.
        # This is the line that makes the workflow eternal without bloat.
        workflow.continue_as_new(args=[day + 1, max_days])


print("✅ Defined DailyReportWorkflow using continue_as_new")

In [ ]:
# NOTE: Temporal normally runs workflow code in a sandbox that re-imports the
# defining module. In a notebook that module is `__main__`, so the re-import
# re-runs these cells and fails with "Failed validating workflow ..." (caused by
# "asyncio.run() cannot be called from a running event loop"). Notebooks must use
# UnsandboxedWorkflowRunner. In a real worker process keep the default sandbox --
# it is what protects you from non-deterministic imports.
from temporalio.worker import UnsandboxedWorkflowRunner
async def demo_continue_as_new():
    async with Worker(
        client,
        task_queue=TASK_QUEUE,
        workflows=[DailyReportWorkflow],
        activities=[run_daily_report],
        workflow_runner=UnsandboxedWorkflowRunner(),
    ):
        return await client.execute_workflow(
            DailyReportWorkflow.run,
            # day=1, max_days=3
            args=[1, 3],
            id=f"daily-report-{uuid.uuid4()}",
            task_queue=TASK_QUEUE,
        )


result = await demo_continue_as_new()
print(f"📬 Final result: {result}")
print("\n💡 In the Temporal UI you'll see 3 separate executions chained together —")
print("   each run links to the 'next' execution. Event history stays tiny forever.")

---
## 🧪 Part 4: Testing — no Docker, no flaky CI

### The problem

You want to run your workflow tests in CI, but spinning up the full Temporal stack (Postgres + server + UI) is slow and fragile.

### The solution: `WorkflowEnvironment`

The Temporal SDK ships a **local in-process test server**. It behaves exactly like real Temporal but starts in ~1 second and runs entirely inside your test process. Two modes:

| Mode | How |
|---|---|
| `start_local()` | Real Temporal server in a subprocess (first run downloads a binary ≈ 30 MB) |
| `start_time_skipping()` | Fake server that **fast-forwards `workflow.sleep()`** — tests that would wait hours finish instantly |

We'll use `start_time_skipping()` so a 1-day timer runs in milliseconds.

In [ ]:
# A workflow we want to test: waits 1 day, then runs an activity.

@activity.defn
async def send_birthday_email(user_id: str) -> str:
    return f"Sent birthday email to {user_id}"


@workflow.defn
class BirthdayWorkflow:
    @workflow.run
    async def run(self, user_id: str) -> str:
        # Real-world: wait until the user's birthday (could be days away)
        await workflow.sleep(timedelta(days=1))
        return await workflow.execute_activity(
            send_birthday_email, user_id,
            start_to_close_timeout=timedelta(seconds=10),
        )


print("✅ Defined BirthdayWorkflow (waits 1 day before sending email)")

In [ ]:
# NOTE: Temporal normally runs workflow code in a sandbox that re-imports the
# defining module. In a notebook that module is `__main__`, so the re-import
# re-runs these cells and fails with "Failed validating workflow ..." (caused by
# "asyncio.run() cannot be called from a running event loop"). Notebooks must use
# UnsandboxedWorkflowRunner. In a real worker process keep the default sandbox --
# it is what protects you from non-deterministic imports.
from temporalio.worker import UnsandboxedWorkflowRunner
# Test it using WorkflowEnvironment.start_time_skipping() — the 1-day sleep
# completes in milliseconds because the fake server skips over timers.

from temporalio.testing import WorkflowEnvironment

async def test_birthday_workflow():
    async with await WorkflowEnvironment.start_time_skipping() as env:
        async with Worker(
            env.client,
            task_queue="test-queue",
            workflows=[BirthdayWorkflow],
            activities=[send_birthday_email],
            workflow_runner=UnsandboxedWorkflowRunner(),
        ):
            result = await env.client.execute_workflow(
                BirthdayWorkflow.run,
                "user-123",
                id="test-birthday",
                task_queue="test-queue",
            )
            return result


result = await test_birthday_workflow()
assert result == "Sent birthday email to user-123", f"Unexpected: {result}"
print(f"✅ Test passed: {result}")
print("   A 24-hour timer ran in milliseconds thanks to time-skipping!")

### Mocking activities in tests

You can also swap an activity implementation at test time — handy when the real one calls Stripe:

```python
@activity.defn(name="send_birthday_email")   # same name as production activity
async def fake_send_birthday_email(user_id: str) -> str:
    return f"MOCK-email-{user_id}"

# Register the fake in your test Worker instead of the real one.
```

Now your workflow test runs with zero external dependencies.

## 📚 Summary

| Feature | API | Why it matters |
|---|---|---|
| **Query** | `@workflow.query` + `handle.query(...)` | Live read-only state for dashboards and support tools |
| **Heartbeat** | `activity.heartbeat(...)` + `heartbeat_timeout` | Detect dead workers fast; resume long activities from last progress |
| **Continue-As-New** | `workflow.continue_as_new(args=[...])` | Run forever without history bloat |
| **Testing** | `WorkflowEnvironment.start_time_skipping()` | Run workflow tests in CI without Docker and without real waits |

### Mental model

```
            ┌─── signal  (write) ──▶│
  Outside  ─┼─── query   (read)  ──▶│ Workflow ──▶ Activity ─── heartbeat ──▶ Temporal
            │                       │                                    ("I'm still alive")
            └─── result  (final) ◀──│
                                    │
                           continue_as_new
                                    │
                                    ▼
                            Fresh execution
```

### Where to go next

- **Worker tuning** — concurrent activity/workflow task slots, rate limiting per task queue
- **Versioning** — `workflow.patched()` for safely changing workflow code while old ones are still running
- **Observability** — `temporalio.runtime` Prometheus metrics, OpenTelemetry tracing
- **Search attributes** — index workflows by custom fields (customer_id, region, …) for fast filtering in the UI
- **Schedules** — Temporal-native cron replacement (see `client.create_schedule(...)`)